# 2º Trabalho de Fundamentos de IA — Etapa 3: Probabilidade
**Domínio:** Educação (SAEB 2021/2023 + Censo 2010) · **IComp/UFAM** · Prof. Edjard Mota

Esta etapa é de **cálculo e interpretação** (sem modelos complexos). Cobre as 5 tarefas do enunciado:

1. Duas variáveis binárias → distribuição conjunta (4 eventos atômicos)
2. Probabilidades marginais e condicionais
3. Regra da cadeia para 3 variáveis
4. Teorema de Bayes em um caso real do domínio (causal × diagnóstico)
5. Independência condicional entre 2 atributos dada a classe

E ainda a discussão do gabarito: *por que a lógica de primeira ordem (LPO) não expressa "85% de chance"*.

> As probabilidades abaixo são **frequências relativas** no conjunto de 3.519 exemplos (município × dependência × localização) usado nas Etapas 1 e 2. Como um mesmo município pode aparecer em mais de uma linha, os exemplos não são totalmente independentes; por isso evitamos tirar conclusões de testes estatísticos finos e olhamos a **magnitude** das diferenças.

## 0. Dados e variáveis binárias

In [1]:
import io, gzip, base64
import numpy as np, pandas as pd
pd.set_option('display.width', 140)

EMBUTIDO = "..."
df = pd.read_csv(io.StringIO(gzip.decompress(base64.b64decode(EMBUTIDO)).decode()))

# Três variáveis binárias (1 = verdadeiro)
df['M'] = (df.MEDIA_12_MT_2023 > df.MEDIA_12_MT_2021).astype(int)   # M: a média de Matemática MELHOROU (alvo da Etapa 1)
df['B'] = (df.MEDIA_12_MT_2021 < 265).astype(int)                   # B: média inicial BAIXA (2021 < 265 pontos)
df['I'] = (df.IDHM_R < 0.6).astype(int)                             # I: IDHM-Renda BAIXO (2010, < 0,6)
N = len(df)
print(f'N = {N} exemplos')
print(df[['M', 'B', 'I']].mean().round(3).rename({'M': 'P(M=1)', 'B': 'P(B=1)', 'I': 'P(I=1)'}).to_string())

N = 3519 exemplos
P(M=1)    0.631
P(B=1)    0.483
P(I=1)    0.327


## 1. Distribuição conjunta de duas variáveis (M, B)
Quatro **eventos atômicos**: (M,B) ∈ {(0,0), (0,1), (1,0), (1,1)}. Cada probabilidade é a contagem do evento dividida por N. A soma tem de ser 1.

In [2]:
cont = pd.crosstab(df.M, df.B)
conj = cont / N
conj.index = ['M=0 (não melhorou)', 'M=1 (melhorou)']; conj.columns = ['B=0 (inicial alta)', 'B=1 (inicial baixa)']
print(conj.round(4).to_string())
print(f'\nSoma da conjunta = {conj.values.sum():.4f}')

                    B=0 (inicial alta)  B=1 (inicial baixa)
M=0 (não melhorou)              0.2944               0.0745
M=1 (melhorou)                  0.2225               0.4086

Soma da conjunta = 1.0000


## 2. Marginais e condicionais
- **Marginal:** soma sobre a outra variável, ex.: $P(M=1)=P(M=1,B=0)+P(M=1,B=1)$.
- **Condicional:** $P(A\mid C)=\dfrac{P(A,C)}{P(C)}$.

In [3]:
P = lambda m=None, b=None: (conj.values[m, b] if (m is not None and b is not None)
                            else conj.values[m, :].sum() if m is not None else conj.values[:, b].sum())
print('Marginais')
print(f'  P(M=1) = {P(m=1):.4f}   P(B=1) = {P(b=1):.4f}')
print('\nCondicionais')
print(f'  P(M=1 | B=1) = P(1,1)/P(B=1) = {P(1,1):.4f}/{P(b=1):.4f} = {P(1,1)/P(b=1):.4f}')
print(f'  P(M=1 | B=0) = P(1,0)/P(B=0) = {P(1,0):.4f}/{P(b=0):.4f} = {P(1,0)/P(b=0):.4f}')
print(f'  P(B=1 | M=1) = P(1,1)/P(M=1) = {P(1,1):.4f}/{P(m=1):.4f} = {P(1,1)/P(m=1):.4f}')
print(f'  P(B=1 | M=0) = P(0,1)/P(M=0) = {P(0,1):.4f}/{P(m=0):.4f} = {P(0,1)/P(m=0):.4f}')
print(f'\nM e B são independentes? P(M=1,B=1) = {P(1,1):.4f}  vs  P(M=1)·P(B=1) = {P(m=1)*P(b=1):.4f}  → NÃO')

Marginais
  P(M=1) = 0.6311   P(B=1) = 0.4831

Condicionais
  P(M=1 | B=1) = P(1,1)/P(B=1) = 0.4086/0.4831 = 0.8459
  P(M=1 | B=0) = P(1,0)/P(B=0) = 0.2225/0.5169 = 0.4305
  P(B=1 | M=1) = P(1,1)/P(M=1) = 0.4086/0.6311 = 0.6475
  P(B=1 | M=0) = P(0,1)/P(M=0) = 0.0745/0.3689 = 0.2018

M e B são independentes? P(M=1,B=1) = 0.4086  vs  P(M=1)·P(B=1) = 0.3049  → NÃO


**Leitura:** a chance de melhorar é de ~85% para quem partiu de média baixa e de ~43% para quem partiu de média alta. As duas variáveis são claramente **dependentes** (regressão à média, já vista nas Etapas 1 e 2).

## 3. Regra da cadeia com 3 variáveis (B, I, M)
$$P(B,I,M)=P(B)\;P(I\mid B)\;P(M\mid B,I)$$
Calculamos cada fator separadamente e conferimos que o produto reproduz a conjunta dos 8 eventos atômicos.

In [4]:
conj3 = df.groupby(['B', 'I', 'M']).size() / N          # conjunta observada (8 eventos)
pB = df.B.value_counts(normalize=True)                     # P(B)
pI_B = df.groupby('B').I.value_counts(normalize=True)      # P(I | B)
pM_BI = df.groupby(['B', 'I']).M.value_counts(normalize=True)  # P(M | B, I)

linhas = []
for (b, i, m), p in conj3.items():
    prod = pB[b] * pI_B[(b, i)] * pM_BI[(b, i, m)]
    linhas.append(dict(B=b, I=i, M=m, **{'P(B)': pB[b], 'P(I|B)': pI_B[(b, i)], 'P(M|B,I)': pM_BI[(b, i, m)],
                                         'produto': prod, 'conjunta': p}))
cad = pd.DataFrame(linhas); print(cad.round(4).to_string(index=False))
print(f"\nSoma = {cad.conjunta.sum():.4f} | máx |produto − conjunta| = {(cad['produto'] - cad.conjunta).abs().max():.1e}")

 B  I  M   P(B)  P(I|B)  P(M|B,I)  produto  conjunta
 0  0  0 0.5169  0.8411    0.5824   0.2532    0.2532
 0  0  1 0.5169  0.8411    0.4176   0.1816    0.1816
 0  1  0 0.5169  0.1589    0.5017   0.0412    0.0412
 0  1  1 0.5169  0.1589    0.4983   0.0409    0.0409
 1  0  0 0.4831  0.4941    0.1952   0.0466    0.0466
 1  0  1 0.4831  0.4941    0.8048   0.1921    0.1921
 1  1  0 0.4831  0.5059    0.1140   0.0278    0.0278
 1  1  1 0.4831  0.5059    0.8860   0.2165    0.2165

Soma = 1.0000 | máx |produto − conjunta| = 1.4e-17


O produto reproduz exatamente a conjunta, como a regra da cadeia garante (vale para **qualquer** ordem das variáveis, sem exigir independência). O ganho prático aparece nas Etapas 4 e 5: se algumas variáveis forem independentes dadas outras, alguns fatores se simplificam e precisamos de menos parâmetros.

## 4. Teorema de Bayes em um caso real: causal × diagnóstico
Ideia: dado que um município **melhorou** (M=1), qual a probabilidade de ele ter **partido de média baixa** (B=1)?

$$P(B\mid M)=\frac{P(M\mid B)\,P(B)}{P(M)},\qquad P(M)=P(M\mid B)P(B)+P(M\mid\neg B)P(\neg B)$$

- **Direção causal:** $P(M\mid B)$ — "partindo de média baixa, quanto se melhora?". Segue o sentido do tempo (2021 → 2023) e é o que o mecanismo (regressão à média, espaço para subir) explica.
- **Direção diagnóstica:** $P(B\mid M)$ — "vendo que melhorou, o que se infere sobre o ponto de partida?". Raciocina do efeito para a causa; é a pergunta de quem só observa o resultado.

In [5]:
p_M_B  = P(1,1) / P(b=1)      # causal:  P(M=1 | B=1)
p_M_nB = P(1,0) / P(b=0)      # causal:  P(M=1 | B=0)
p_B    = P(b=1)               # a priori: P(B=1)
p_M    = p_M_B * p_B + p_M_nB * (1 - p_B)          # evidência (prob. total)
p_B_M  = p_M_B * p_B / p_M                          # diagnóstica via Bayes

print(f'P(M|B) = {p_M_B:.4f}   P(M|¬B) = {p_M_nB:.4f}   P(B) = {p_B:.4f}')
print(f'P(M) = {p_M_B:.4f}·{p_B:.4f} + {p_M_nB:.4f}·{1-p_B:.4f} = {p_M:.4f}   (confere com a marginal: {P(m=1):.4f})')
print(f'\nBayes: P(B|M) = {p_M_B:.4f}·{p_B:.4f} / {p_M:.4f} = {p_B_M:.4f}')
print(f'Contagem direta: {int(cont.loc[1,1])}/{int(cont.loc[1].sum())} = {cont.loc[1,1]/cont.loc[1].sum():.4f}')
print(f'\nSem olhar o resultado, a chance de partir de média baixa é {p_B:.1%}; sabendo que melhorou, sobe para {p_B_M:.1%}.')

P(M|B) = 0.8459   P(M|¬B) = 0.4305   P(B) = 0.4831
P(M) = 0.8459·0.4831 + 0.4305·0.5169 = 0.6311   (confere com a marginal: 0.6311)

Bayes: P(B|M) = 0.8459·0.4831 / 0.6311 = 0.6475
Contagem direta: 1438/2221 = 0.6475

Sem olhar o resultado, a chance de partir de média baixa é 48.3%; sabendo que melhorou, sobe para 64.7%.


**Interpretação:** saber que o município melhorou aumenta a chance de ele ter partido de uma média baixa (de ~48% para ~65%). Atenção: isso é **inferência diagnóstica** (correlação observada), e **não** significa que "melhorar causa média baixa". Também não prova efeito do Novo Ensino Médio: o mesmo padrão aparece por regressão à média.

## 5. Independência condicional dada a classe
Em Naive Bayes (Etapa 4), assume-se que os atributos são independentes **dada a classe**: $P(B,I\mid M)=P(B\mid M)\,P(I\mid M)$. Verificamos para os atributos **B** (média inicial baixa) e **I** (IDHM baixo), em cada valor de M.

In [6]:
linhas = []
for m in (0, 1):
    d = df[df.M == m]; n = len(d)
    pB_m, pI_m = d.B.mean(), d.I.mean()
    for b in (0, 1):
        for i in (0, 1):
            conj_obs = ((d.B == b) & (d.I == i)).mean()
            prod = (pB_m if b else 1 - pB_m) * (pI_m if i else 1 - pI_m)
            linhas.append(dict(M=m, B=b, I=i, **{'P(B,I|M) observado': conj_obs, 'P(B|M)·P(I|M)': prod, 'diferença': conj_obs - prod}))
ci = pd.DataFrame(linhas); print(ci.round(3).to_string(index=False))
print('\nMaior diferença absoluta:', round(float(ci['diferença'].abs().max()), 3))

# para comparação: independência marginal entre B e I (sem condicionar)
marg = pd.crosstab(df.B, df.I, normalize=True).values
print(f"\nSem condicionar: P(B=1,I=1) = {marg[1,1]:.3f}  vs  P(B=1)·P(I=1) = {df.B.mean()*df.I.mean():.3f}")

 M  B  I  P(B,I|M) observado  P(B|M)·P(I|M)  diferença
 0  0  0               0.686          0.649      0.038
 0  0  1               0.112          0.149     -0.038
 0  1  0               0.126          0.164     -0.038
 0  1  1               0.076          0.038      0.038
 1  0  0               0.288          0.209      0.079
 1  0  1               0.065          0.144     -0.079
 1  1  0               0.304          0.383     -0.079
 1  1  1               0.343          0.264      0.079

Maior diferença absoluta: 0.079

Sem condicionar: P(B=1,I=1) = 0.244  vs  P(B=1)·P(I=1) = 0.158


**Resultado:** B e I **não** são condicionalmente independentes dado M. Em M=1, por exemplo, o par (B=1, I=1) ocorre em ~34% dos casos, contra ~26% se fossem independentes (diferença de ~8 pontos). Faz sentido: municípios de renda baixa tendem a ter média de Matemática baixa, **mesmo sabendo se melhoraram ou não**. Consequência para a Etapa 4: o Naive Bayes vai **violar** essa suposição para esses dois atributos; é uma limitação a discutir (e uma razão para a Rede Bayesiana da Etapa 5, que permite modelar a dependência entre atributos).

## 6. Por que a LPO não expressa "85% de chance"?
Em lógica de primeira ordem, uma regra como $\forall x\,(\text{BaixaInicial}(x)\rightarrow \text{Melhora}(x))$ é **verdadeira ou falsa**. Basta **um** contraexemplo para ela ser falsa, e a lógica não tem como dizer "vale na maioria dos casos". Veja nos dados:

In [7]:
n_B = int((df.B == 1).sum()); contra = int(((df.B == 1) & (df.M == 0)).sum())
print(f'Municípios com B=1 (inicial baixa): {n_B}')
print(f'Contraexemplos (B=1 e M=0): {contra}  ({contra/n_B:.1%})')
print(f'Valem a regra (B=1 e M=1):  {n_B-contra}  ({1-contra/n_B:.1%})')
print(f'\nEm LPO: ∀x (B(x) → M(x)) é FALSA (há {contra} contraexemplos).')
print(f'Em probabilidade: P(M=1 | B=1) = {1-contra/n_B:.2f}  — informação útil que a LPO descarta.')

Municípios com B=1 (inicial baixa): 1700
Contraexemplos (B=1 e M=0): 262  (15.4%)
Valem a regra (B=1 e M=1):  1438  (84.6%)

Em LPO: ∀x (B(x) → M(x)) é FALSA (há 262 contraexemplos).
Em probabilidade: P(M=1 | B=1) = 0.85  — informação útil que a LPO descarta.


- **LPO:** a regra é falsa e some; não distingue "falha em 15% dos casos" de "falha em 85%". Tampouco combina evidências incertas nem atualiza uma crença com novos dados.
- **Probabilidade:** representa o **grau de crença** (0 a 1), permite condicionar em evidências ($P(M\mid B)$), inverter o raciocínio (Bayes) e ser aprendida dos dados. É a base das Etapas 4 a 6.

## 7. Resumo
| Item | Resultado |
|---|---|
| Conjunta (M,B) | soma 1; P(M=1,B=1)=0,409 |
| Marginais | P(M=1)=0,631; P(B=1)=0,483 |
| Condicionais | P(M=1\|B=1)=0,846; P(M=1\|B=0)=0,430 |
| Regra da cadeia (B,I,M) | produto = conjunta (8 eventos) |
| Bayes | P(B=1\|M=1)=0,647 (a priori 0,483) |
| Independência condicional B⊥I \| M | **não vale** (dif. máx. ≈ 0,08) |